# Hiệu chỉnh mức chi tiết ảnh

Chạy trên CPU với package `medvqa` và phụ thuộc `api`. Mount Drive nếu cần trước khi cấu hình đường dẫn. Mặc định chỉ kiểm đầu vào và dự toán cho 599 ảnh. Đặt `EXECUTE=True` để gọi tối đa một ảnh; đối chiếu usage, số dư và dashboard trước khi chạy cả tập. Giữ lại thư mục output để tránh gọi lặp.


In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Đặt MEDVQA_SOURCE_ROOT đến thư mục chứa package medvqa.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
import json

from medvqa import detail_probe

INVENTORY = source_root / "resources/detail_probe/image_detail_probe_inputs.json"
OUTPUT_ROOT = Path("outputs/image_detail_probe")
SEARCH_ROOTS = [OUTPUT_ROOT]
BALANCE_USD = None
EXECUTE = False

inventory = json.loads(INVENTORY.read_text(encoding="utf-8"))
detail_probe.validate_inputs(inventory)
if BALANCE_USD is None:
    BALANCE_USD = float(input("Số dư API hiện tại (USD): "))
forecast = detail_probe.estimate(inventory, BALANCE_USD)
print(json.dumps(forecast, ensure_ascii=False, indent=2))
print("Lần gọi trước:", detail_probe.existing_probes(SEARCH_ROOTS))

In [ ]:
import getpass

if EXECUTE:
    api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")
    try:
        result = detail_probe.run_probe(
            inventory, OUTPUT_ROOT, BALANCE_USD, api_key, SEARCH_ROOTS
        )
    finally:
        api_key = None
    print(json.dumps(result, ensure_ascii=False, indent=2))